# GCS → Kaggle AIC Keyframe Dataset (fast, complete, metadata-safe)

Mục tiêu:

1. Đọc **keyframes + map-keyframes gốc** từ Google Cloud Storage.
2. Tự nhận diện batch `L21` → `L30` và video `Lxx_Vyyy`.
3. Chuẩn hóa output theo layout dùng trực tiếp bởi notebook SigLIP2:

```text
001_Keyframes_L21/
├── keyframes/
│   ├── L21_V001/
│   │   ├── 001.jpg
│   │   └── ...
│   └── ...
└── map-keyframes/
    ├── L21_V001.csv
    └── ...

002_Keyframes_L22/
...
010_Keyframes_L30/
```

4. **Không tái tạo map-keyframes**: CSV được tải nguyên bản từ GCS.
5. Validate:
   - mỗi video có đúng 1 map CSV;
   - số dòng map khớp số keyframe;
   - `n` khớp tên frame số;
   - kiểm tra `pts_time`, `fps`, `frame_idx`;
   - tạo `frames_metadata.parquet` để join với vector sau này.
6. Download song song nhiều file nhỏ bằng Google Cloud Storage Transfer Manager.
7. Upload lên Kaggle bằng `--dir-mode tar`: tar chỉ là cơ chế vận chuyển thư mục nhanh/không nén.

> Security: notebook không hard-code GCS/Milvus/Kaggle token. Dùng Kaggle Secrets.


## 1. Install dependencies


In [ ]:
%pip install -q -U google-cloud-storage kaggle pandas pyarrow pillow tqdm


## 2. Configuration


In [ ]:
from pathlib import Path
import os
import re
import json
import math
import shutil

BATCHES = [f"L{i}" for i in range(21, 31)]
BATCH_ORDER = {batch: i + 1 for i, batch in enumerate(BATCHES)}

# ---------- GCS ----------
GCS_BUCKET_FALLBACK = "aic_ai_2026"
GCS_BUCKET_SECRET_NAME = "GCS_BUCKET"
GCS_CREDENTIALS_SECRET_NAMES = [
    "GCS_CREDENTIALS_JSON",
    "GCS_SERVICE_ACCOUNT_JSON",
]

# [] = scan toàn bucket một lần.
# Nếu biết prefix chính xác, điền vào đây để scan nhanh hơn.
GCS_PREFIXES = []

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

FRAME_PATH_MARKERS = {"keyframes", "frames"}
MAP_PATH_MARKERS = {
    "map-keyframes",
    "map_keyframes",
    "mapkeyframes",
    "map-keyframe",
    "map_keyframe",
}
FALLBACK_EXCLUDED_PARTS = {
    "features", "feature", "embeddings", "embedding",
    "captions", "caption", "ocr", "objects", "object",
    "thumbnails", "thumbnail",
}

# ---------- PERFORMANCE ----------
CPU_COUNT = os.cpu_count() or 4
MAX_WORKERS = min(64, max(24, CPU_COUNT * 8))
TRANSFER_BATCH = 4000
MAX_RETRIES = 3
RESUME = True

# ---------- LOCAL OUTPUT ----------
DATASET_DIR = Path("/kaggle/working/aic_ai_2026_keyframes")
MANIFEST_PATH = DATASET_DIR / "dataset_manifest.parquet"
FRAME_METADATA_PATH = DATASET_DIR / "frames_metadata.parquet"
SUMMARY_PATH = DATASET_DIR / "dataset_summary.csv"
VALIDATION_PATH = DATASET_DIR / "validation_report.csv"

# ---------- VALIDATION ----------
REQUIRE_MAP_FOR_EVERY_VIDEO = True
STRICT_MAP_SCHEMA = True
EXPECTED_MAP_COLUMNS = {"n", "pts_time", "fps", "frame_idx"}

# ---------- KAGGLE ----------
# Owner mặc định lấy từ INPUT_ROOT trong notebook embedding bạn gửi.
# Đổi nếu tài khoản Kaggle thực tế khác.
KAGGLE_DATASET_OWNER = "khngxuninh"
KAGGLE_DATASET_SLUG = "aic-ai-2026-keyframes-l21-l30"
KAGGLE_DATASET_ID = f"{KAGGLE_DATASET_OWNER}/{KAGGLE_DATASET_SLUG}"
KAGGLE_DATASET_TITLE = "AIC AI 2026 Keyframes L21-L30"

# Save Version / Run All sẽ upload sau khi validation PASS.
RUN_KAGGLE_UPLOAD = True
KAGGLE_PUBLIC = False

# Chừa safety margin dưới giới hạn Kaggle dataset.
MAX_KAGGLE_DATASET_GB = 195.0

print("CPU cores:", CPU_COUNT)
print("GCS download workers:", MAX_WORKERS)
print("Batches:", BATCHES)
print("Kaggle Dataset ID:", KAGGLE_DATASET_ID)


## 3. Load Kaggle Secrets and authenticate GCS

Secrets:

- `GCS_BUCKET` (optional nếu bucket là `aic_ai_2026`)
- `GCS_CREDENTIALS_JSON` (**recommended**, cùng tên notebook embedding)
- hoặc `GCS_SERVICE_ACCOUNT_JSON`
- `KAGGLE_API_TOKEN` nếu Kaggle CLI cần token
- hoặc legacy `KAGGLE_USERNAME` + `KAGGLE_KEY`

Milvus/Zilliz credentials không cần cho notebook transfer.


In [ ]:
from google.cloud import storage

def get_kaggle_secret_optional(name: str):
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
        return value if value else None
    except Exception:
        return os.environ.get(name)

def first_secret(names):
    for name in names:
        value = get_kaggle_secret_optional(name)
        if value:
            return name, value
    return None, None

bucket_secret = get_kaggle_secret_optional(GCS_BUCKET_SECRET_NAME)
GCS_BUCKET = bucket_secret or GCS_BUCKET_FALLBACK

credential_name, credential_text = first_secret(GCS_CREDENTIALS_SECRET_NAMES)
if not credential_text:
    raise RuntimeError(
        "Missing GCS credentials. Add Kaggle Secret GCS_CREDENTIALS_JSON "
        "(recommended) or GCS_SERVICE_ACCOUNT_JSON."
    )

service_account_info = json.loads(credential_text)
gcs_client = storage.Client.from_service_account_info(service_account_info)
bucket = gcs_client.bucket(GCS_BUCKET)

_ = next(iter(gcs_client.list_blobs(GCS_BUCKET, max_results=1)), None)

# Kaggle auth: do not print secret values.
kaggle_api_token = get_kaggle_secret_optional("KAGGLE_API_TOKEN")
if kaggle_api_token:
    os.environ["KAGGLE_API_TOKEN"] = kaggle_api_token

legacy_user = get_kaggle_secret_optional("KAGGLE_USERNAME")
legacy_key = get_kaggle_secret_optional("KAGGLE_KEY")
if legacy_user and legacy_key:
    os.environ["KAGGLE_USERNAME"] = legacy_user
    os.environ["KAGGLE_KEY"] = legacy_key

print("Connected GCS bucket:", GCS_BUCKET)
print("GCS project:", gcs_client.project)
print("Credentials source:", credential_name)
print("Kaggle auth secret detected:", bool(kaggle_api_token or (legacy_user and legacy_key)))


## 4. Scan GCS once and build canonical manifest

Auto-detect:

- frame chứa video id dạng `Lxx_Vyyy`;
- ưu tiên path có `keyframes`;
- sau đó `frames`;
- fallback chỉ dùng image có video id và loại các output path rõ ràng (`features`, `embeddings`, `ocr`, ...);
- map CSV phải chứa video id và marker `map-keyframes`/biến thể.

Nếu có nhiều object map vào cùng một destination, notebook dừng để tránh lấy nhầm dữ liệu.


In [ ]:
from collections import Counter
from pathlib import PurePosixPath
import pandas as pd
from tqdm.auto import tqdm

VIDEO_RE = re.compile(r"(?i)(L\d{2}_V\d+)")

def safe_gcs_name(name: str) -> bool:
    p = PurePosixPath(name)
    return (not p.is_absolute()) and (".." not in p.parts)

def normalize_video_id(value: str) -> str:
    m = VIDEO_RE.search(value)
    return m.group(1).upper() if m else ""

def batch_from_video(video_id: str) -> str:
    return video_id.split("_", 1)[0].upper() if video_id else ""

def path_parts_lower(name: str):
    return [p.lower() for p in PurePosixPath(name).parts if p]

def frame_candidate_score(name: str) -> int:
    parts = path_parts_lower(name)
    if "keyframes" in parts:
        return 30
    if any("keyframes_" in p or p.startswith("keyframes") for p in parts):
        return 25
    if "frames" in parts:
        return 10
    if any(p in FALLBACK_EXCLUDED_PARTS for p in parts):
        return -1
    return 0

def is_map_candidate(name: str) -> bool:
    if Path(name).suffix.lower() != ".csv":
        return False
    parts = path_parts_lower(name)
    joined = "/".join(parts)
    return any(p in MAP_PATH_MARKERS for p in parts) or ("map" in joined and "keyframe" in joined)

def canonical_batch_dir(batch_id: str) -> str:
    return f"{BATCH_ORDER[batch_id]:03d}_Keyframes_{batch_id}"

def canonical_frame_path(batch_id: str, video_id: str, filename: str) -> str:
    return f"{canonical_batch_dir(batch_id)}/keyframes/{video_id}/{filename}"

def canonical_map_path(batch_id: str, video_id: str) -> str:
    return f"{canonical_batch_dir(batch_id)}/map-keyframes/{video_id}.csv"

def scan_gcs(prefixes=None):
    prefixes = prefixes or [""]
    frame_candidates = []
    map_candidates = []
    counters = Counter()
    total_objects = 0
    total_bytes = 0

    for prefix in prefixes:
        iterator = gcs_client.list_blobs(GCS_BUCKET, prefix=prefix, page_size=1000)
        for blob in tqdm(iterator, desc=f"Scanning gs://{GCS_BUCKET}/{prefix}"):
            name = blob.name
            if not safe_gcs_name(name):
                continue

            total_objects += 1
            size = int(blob.size or 0)
            total_bytes += size

            video_id = normalize_video_id(name)
            if not video_id:
                continue
            batch_id = batch_from_video(video_id)
            if batch_id not in BATCHES:
                continue

            suffix = Path(name).suffix.lower()
            base = {
                "source_name": name,
                "source_gcs_uri": f"gs://{GCS_BUCKET}/{name}",
                "size_bytes": size,
                "generation": int(blob.generation) if blob.generation else None,
                "md5_hash": blob.md5_hash,
                "content_type": blob.content_type,
                "batch_id": batch_id,
                "batch_order": BATCH_ORDER[batch_id],
                "video_id": video_id,
            }

            if suffix in IMAGE_EXTENSIONS:
                score = frame_candidate_score(name)
                if score >= 0:
                    row = dict(base)
                    row["kind"] = "frame"
                    row["source_score"] = score
                    row["frame_filename"] = Path(name).name
                    stem = Path(name).stem
                    row["frame_number"] = int(stem) if stem.isdigit() else None
                    frame_candidates.append(row)
                    counters[f"frame_score_{score}"] += 1

            elif is_map_candidate(name):
                row = dict(base)
                row["kind"] = "map"
                row["source_score"] = 30
                row["frame_filename"] = None
                row["frame_number"] = None
                map_candidates.append(row)
                counters["maps"] += 1

    frames = pd.DataFrame(frame_candidates)
    maps = pd.DataFrame(map_candidates)
    if frames.empty:
        raise RuntimeError("No frame candidates detected in target batches.")

    selected_frames = []
    chosen_scores = {}
    for batch_id in BATCHES:
        bdf = frames[frames["batch_id"] == batch_id]
        if bdf.empty:
            continue
        best_score = int(bdf["source_score"].max())
        chosen_scores[batch_id] = best_score
        selected_frames.append(bdf[bdf["source_score"] == best_score])

    frames = pd.concat(selected_frames, ignore_index=True) if selected_frames else pd.DataFrame()

    frames["dest_rel_path"] = frames.apply(
        lambda r: canonical_frame_path(r["batch_id"], r["video_id"], r["frame_filename"]),
        axis=1,
    )

    if not maps.empty:
        maps["dest_rel_path"] = maps.apply(
            lambda r: canonical_map_path(r["batch_id"], r["video_id"]),
            axis=1,
        )

    manifest = pd.concat([frames, maps], ignore_index=True, sort=False)
    manifest = manifest.sort_values(
        ["batch_order", "video_id", "kind", "frame_number", "dest_rel_path"],
        na_position="last",
    ).reset_index(drop=True)

    dup = manifest[manifest.duplicated("dest_rel_path", keep=False)]
    if not dup.empty:
        display(
            dup[
                ["kind", "batch_id", "video_id", "source_name", "dest_rel_path", "source_score"]
            ].head(50)
        )
        raise RuntimeError(
            "Duplicate canonical destination paths detected. "
            "Set GCS_PREFIXES to the intended source tree."
        )

    scan_info = {
        "total_objects_scanned": total_objects,
        "total_bucket_bytes_seen": total_bytes,
        "selected_files": len(manifest),
        "selected_bytes": int(manifest["size_bytes"].sum()) if not manifest.empty else 0,
        "chosen_frame_scores": chosen_scores,
    }
    return manifest, scan_info, counters

manifest_df, scan_info, scan_counters = scan_gcs(GCS_PREFIXES)

print(json.dumps(scan_info, indent=2))
print("\nCounters:", dict(scan_counters))
display(
    manifest_df[
        ["kind", "batch_id", "video_id", "source_name", "dest_rel_path", "size_bytes"]
    ].head(30)
)


## 5. Source completeness + storage/time estimate

Dừng nếu thiếu map-keyframes hoặc vượt ngưỡng an toàn gần giới hạn Kaggle.


In [ ]:
summary_df = (
    manifest_df.groupby(["batch_id", "batch_order", "kind"], as_index=False)
    .agg(files=("dest_rel_path", "count"), bytes=("size_bytes", "sum"))
)
summary_df["GB"] = summary_df["bytes"] / 1024**3

frame_videos = (
    manifest_df[manifest_df["kind"] == "frame"]
    .groupby(["batch_id", "video_id"])
    .size()
    .rename("frame_count")
    .reset_index()
)
map_videos = (
    manifest_df[manifest_df["kind"] == "map"]
    .groupby(["batch_id", "video_id"])
    .size()
    .rename("map_count")
    .reset_index()
)

coverage = frame_videos.merge(map_videos, on=["batch_id", "video_id"], how="left")
coverage["map_count"] = coverage["map_count"].fillna(0).astype(int)

missing_maps = coverage[coverage["map_count"] == 0]
duplicate_maps = coverage[coverage["map_count"] > 1]

display(summary_df)
display(coverage.head(20))

if REQUIRE_MAP_FOR_EVERY_VIDEO and not missing_maps.empty:
    display(missing_maps.head(50))
    raise RuntimeError(f"{len(missing_maps)} videos have frames but no map-keyframes CSV.")

if not duplicate_maps.empty:
    display(duplicate_maps.head(50))
    raise RuntimeError(f"{len(duplicate_maps)} videos have multiple map-keyframes CSV files.")

total_bytes = int(manifest_df["size_bytes"].sum())
total_gb = total_bytes / 1024**3
frame_count = int((manifest_df["kind"] == "frame").sum())
map_count = int((manifest_df["kind"] == "map").sum())
video_count = int(frame_videos.shape[0])

if total_gb > MAX_KAGGLE_DATASET_GB:
    raise RuntimeError(
        f"Selected payload is {total_gb:.2f} GB, above safety threshold "
        f"{MAX_KAGGLE_DATASET_GB:.1f} GB. Split the Kaggle Dataset."
    )

free_gb = shutil.disk_usage("/kaggle/working").free / 1024**3
largest_batch_gb = manifest_df.groupby("batch_id")["size_bytes"].sum().max() / 1024**3

print(f"Videos: {video_count:,}")
print(f"Frames: {frame_count:,}")
print(f"Map CSVs: {map_count:,}")
print(f"Selected payload: {total_gb:.2f} GB")
print(f"Largest batch: {largest_batch_gb:.2f} GB")
print(f"Free /kaggle/working: {free_gb:.2f} GB")

if free_gb < total_gb * 1.05:
    print(
        "\nWARNING: free disk is close to/below selected payload. "
        "The full raw hierarchy may not fit in this Kaggle session."
    )

rates_mib_s = [20, 40, 80]
est_rows = []
for rate in rates_mib_s:
    seconds_one_way = total_bytes / (rate * 1024**2)
    est_rows.append({
        "assumed_effective_MiB_s": rate,
        "GCS_download_min": seconds_one_way / 60,
        "Kaggle_upload_min_rough": seconds_one_way / 60,
        "two_way_transfer_min": 2 * seconds_one_way / 60,
    })
display(pd.DataFrame(est_rows).round(1))


## 6. Save manifest and create canonical directories


In [ ]:
DATASET_DIR.mkdir(parents=True, exist_ok=True)

for batch_id in BATCHES:
    batch_root = DATASET_DIR / canonical_batch_dir(batch_id)
    (batch_root / "keyframes").mkdir(parents=True, exist_ok=True)
    (batch_root / "map-keyframes").mkdir(parents=True, exist_ok=True)

for _, row in frame_videos.iterrows():
    (
        DATASET_DIR
        / canonical_batch_dir(row["batch_id"])
        / "keyframes"
        / row["video_id"]
    ).mkdir(parents=True, exist_ok=True)

manifest_df.to_parquet(MANIFEST_PATH, index=False)
summary_df.to_csv(SUMMARY_PATH, index=False)

print("Saved:", MANIFEST_PATH)
print("Saved:", SUMMARY_PATH)


## 7. Fast parallel GCS download with retry + resume

Dùng `transfer_manager.download_many()` để map source GCS bất kỳ sang destination AIC chuẩn.


In [ ]:
import time
from google.cloud.storage import transfer_manager
from tqdm.auto import tqdm

def chunk_dataframe(df: pd.DataFrame, n: int):
    for start in range(0, len(df), n):
        yield df.iloc[start:start+n]

def local_path_for_row(row) -> Path:
    return DATASET_DIR.joinpath(*PurePosixPath(row.dest_rel_path).parts)

def is_local_complete(row) -> bool:
    p = local_path_for_row(row)
    try:
        return p.is_file() and p.stat().st_size == int(row.size_bytes)
    except OSError:
        return False

def pending_rows(df: pd.DataFrame) -> pd.DataFrame:
    if not RESUME:
        return df.copy()

    keep = []
    for row in tqdm(df.itertuples(index=False), total=len(df), desc="Resume check"):
        if not is_local_complete(row):
            p = local_path_for_row(row)
            if p.exists():
                try:
                    p.unlink()
                except OSError:
                    pass
            keep.append(row._asdict())

    return pd.DataFrame(keep, columns=df.columns)

def download_batch_rows(batch_df: pd.DataFrame):
    current = batch_df.copy()

    for attempt in range(1, MAX_RETRIES + 1):
        if current.empty:
            return

        rows = list(current.itertuples(index=False))
        pairs = []
        for row in rows:
            dest = local_path_for_row(row)
            dest.parent.mkdir(parents=True, exist_ok=True)
            pairs.append((bucket.blob(row.source_name), str(dest)))

        results = transfer_manager.download_many(
            pairs,
            worker_type=transfer_manager.THREAD,
            max_workers=MAX_WORKERS,
            skip_if_exists=False,
        )

        failed = []
        for row, result in zip(rows, results):
            if isinstance(result, Exception):
                failed.append(row._asdict())
                continue

            dest = local_path_for_row(row)
            if (not dest.exists()) or dest.stat().st_size != int(row.size_bytes):
                failed.append(row._asdict())

        if not failed:
            return

        print(
            f"Retry {attempt}/{MAX_RETRIES}: "
            f"{len(failed):,}/{len(rows):,} failed or size-mismatched"
        )
        current = pd.DataFrame(failed, columns=batch_df.columns)
        time.sleep(min(2 ** attempt, 10))

    if not current.empty:
        display(current[["source_name", "dest_rel_path", "size_bytes"]].head(30))
        raise RuntimeError(
            f"{len(current):,} files still failed after {MAX_RETRIES} retries."
        )

pending_df = pending_rows(manifest_df)
pending_bytes = int(pending_df["size_bytes"].sum()) if not pending_df.empty else 0

print(f"Pending files: {len(pending_df):,}/{len(manifest_df):,}")
print(f"Pending data: {pending_bytes/1024**3:.2f} GB")

t0 = time.time()
progress = tqdm(total=len(pending_df), desc="Downloading GCS → Kaggle working")

for chunk in chunk_dataframe(pending_df, TRANSFER_BATCH):
    download_batch_rows(chunk)
    progress.update(len(chunk))

progress.close()
elapsed = time.time() - t0

if pending_bytes > 0:
    speed = pending_bytes / 1024**2 / max(elapsed, 1e-9)
    print(f"Download elapsed: {elapsed/60:.2f} min")
    print(f"Effective speed: {speed:.1f} MiB/s")
else:
    print("Nothing to download; all files already complete.")


## 8. Validate map-keyframes and build `frames_metadata.parquet`

CSV map gốc không bị sửa. File Parquet tổng hợp được tạo thêm để join với SigLIP2 vectors.


In [ ]:
import numpy as np

validation_rows = []
metadata_parts = []

frame_manifest = manifest_df[manifest_df["kind"] == "frame"].copy()
map_manifest = manifest_df[manifest_df["kind"] == "map"].copy()

grouped_frames = frame_manifest.groupby(["batch_id", "video_id"], sort=True)

for (batch_id, video_id), vframes in tqdm(
    grouped_frames,
    total=grouped_frames.ngroups,
    desc="Validating videos",
):
    vframes = vframes.sort_values(["frame_number", "frame_filename"], na_position="last")
    vmap_rows = map_manifest[
        (map_manifest["batch_id"] == batch_id)
        & (map_manifest["video_id"] == video_id)
    ]

    issues = []

    if len(vmap_rows) != 1:
        issues.append(f"map_count={len(vmap_rows)}")
        validation_rows.append({
            "batch_id": batch_id,
            "video_id": video_id,
            "frames": len(vframes),
            "map_rows": None,
            "status": "FAIL",
            "issues": "; ".join(issues),
        })
        continue

    map_row = next(vmap_rows.itertuples(index=False))
    map_path = local_path_for_row(map_row)

    if not map_path.exists():
        issues.append("map_file_missing_local")
        validation_rows.append({
            "batch_id": batch_id,
            "video_id": video_id,
            "frames": len(vframes),
            "map_rows": None,
            "status": "FAIL",
            "issues": "; ".join(issues),
        })
        continue

    try:
        mdf = pd.read_csv(map_path)
    except Exception as e:
        issues.append(f"csv_read_error={type(e).__name__}")
        validation_rows.append({
            "batch_id": batch_id,
            "video_id": video_id,
            "frames": len(vframes),
            "map_rows": None,
            "status": "FAIL",
            "issues": "; ".join(issues),
        })
        continue

    missing_cols = EXPECTED_MAP_COLUMNS - set(mdf.columns)
    if missing_cols:
        issues.append("missing_columns=" + ",".join(sorted(missing_cols)))

    if len(mdf) != len(vframes):
        issues.append(f"row_count_mismatch map={len(mdf)} frames={len(vframes)}")

    bad_sizes = 0
    for frow in vframes.itertuples(index=False):
        fp = local_path_for_row(frow)
        if (not fp.exists()) or fp.stat().st_size != int(frow.size_bytes):
            bad_sizes += 1
    if bad_sizes:
        issues.append(f"bad_frame_sizes={bad_sizes}")

    if "n" in mdf.columns:
        n_numeric = pd.to_numeric(mdf["n"], errors="coerce")
        if n_numeric.isna().any():
            issues.append("n_not_numeric")
        else:
            n_values = n_numeric.astype(int).tolist()
            if len(set(n_values)) != len(n_values):
                issues.append("duplicate_n")
            frame_numbers = vframes["frame_number"].dropna().astype(int).tolist()
            if len(frame_numbers) == len(vframes) and sorted(frame_numbers) != sorted(n_values):
                issues.append("n_does_not_match_frame_filenames")

    if "pts_time" in mdf.columns:
        pts = pd.to_numeric(mdf["pts_time"], errors="coerce")
        if pts.isna().any():
            issues.append("pts_time_not_numeric")
        elif not pts.is_monotonic_increasing:
            issues.append("pts_time_not_monotonic")

    if "frame_idx" in mdf.columns:
        frame_idx = pd.to_numeric(mdf["frame_idx"], errors="coerce")
        if frame_idx.isna().any():
            issues.append("frame_idx_not_numeric")
        elif not frame_idx.is_monotonic_increasing:
            issues.append("frame_idx_not_monotonic")

    if "fps" in mdf.columns:
        fps = pd.to_numeric(mdf["fps"], errors="coerce")
        if fps.isna().any() or (fps <= 0).any():
            issues.append("fps_invalid")

    validation_rows.append({
        "batch_id": batch_id,
        "video_id": video_id,
        "frames": len(vframes),
        "map_rows": len(mdf),
        "status": "PASS" if not issues else "FAIL",
        "issues": "; ".join(issues),
    })

    if "n" in mdf.columns:
        tmp = mdf.copy()
        tmp["n_join"] = pd.to_numeric(tmp["n"], errors="coerce")

        fmeta = vframes[
            [
                "batch_id", "batch_order", "video_id",
                "frame_number", "frame_filename", "dest_rel_path",
                "source_name", "source_gcs_uri", "size_bytes",
                "generation", "md5_hash", "content_type",
            ]
        ].copy()

        fmeta = fmeta.rename(columns={
            "source_name": "source_frame_gcs_name",
            "source_gcs_uri": "source_frame_gcs_uri",
            "dest_rel_path": "image_rel_path",
        })
        fmeta["n_join"] = pd.to_numeric(fmeta["frame_number"], errors="coerce")

        # Keep canonical image order so future vector rows can be joined deterministically.
        merged = fmeta.merge(tmp, on="n_join", how="left", validate="one_to_one")
        merged = merged.drop(columns=["n_join"])
        merged["source_map_gcs_name"] = map_row.source_name
        merged["source_map_gcs_uri"] = map_row.source_gcs_uri
        metadata_parts.append(merged)

validation_df = pd.DataFrame(validation_rows).sort_values(["batch_id", "video_id"])
validation_df.to_csv(VALIDATION_PATH, index=False)

failures = validation_df[validation_df["status"] != "PASS"]
display(validation_df.groupby(["batch_id", "status"]).size().unstack(fill_value=0))

if STRICT_MAP_SCHEMA and not failures.empty:
    display(failures.head(100))
    raise RuntimeError(
        f"Validation failed for {len(failures)} videos. "
        "Fix source data/selection before creating Kaggle Dataset."
    )

if metadata_parts:
    frames_metadata_df = pd.concat(metadata_parts, ignore_index=True, sort=False)
    preferred = [
        "batch_id", "batch_order", "video_id",
        "n", "frame_filename", "image_rel_path",
        "pts_time", "fps", "frame_idx",
        "source_frame_gcs_name", "source_frame_gcs_uri",
        "source_map_gcs_name", "source_map_gcs_uri",
        "size_bytes", "generation", "md5_hash", "content_type",
    ]
    cols = [c for c in preferred if c in frames_metadata_df.columns]
    cols += [c for c in frames_metadata_df.columns if c not in cols]
    frames_metadata_df = frames_metadata_df[cols]
    sort_cols = [c for c in ["batch_order", "video_id", "n"] if c in frames_metadata_df.columns]
    if sort_cols:
        frames_metadata_df = frames_metadata_df.sort_values(sort_cols).reset_index(drop=True)
    frames_metadata_df.to_parquet(FRAME_METADATA_PATH, index=False)
    print("Saved:", FRAME_METADATA_PATH)
    print("Frame metadata rows:", len(frames_metadata_df))

print("Saved:", VALIDATION_PATH)


## 9. Write README and Kaggle metadata


In [ ]:
readme_text = f"""# {KAGGLE_DATASET_TITLE}

Canonical AIC keyframe dataset transferred from `gs://{GCS_BUCKET}`.

## Layout

For each batch L21-L30:

`NNN_Keyframes_Lxx/keyframes/Lxx_Vyyy/<frame>.jpg`

`NNN_Keyframes_Lxx/map-keyframes/Lxx_Vyyy.csv`

The original map-keyframes CSV files are preserved without rewriting.

## Root metadata

- `dataset_manifest.parquet`: GCS object -> Kaggle relative path audit manifest.
- `frames_metadata.parquet`: one row per keyframe joined with original temporal map metadata.
- `dataset_summary.csv`: counts and bytes per batch/file kind.
- `validation_report.csv`: validation result per video.

Expected temporal mapping fields: `n`, `pts_time`, `fps`, `frame_idx`.
"""
(DATASET_DIR / "README.md").write_text(readme_text, encoding="utf-8")

metadata = {
    "title": KAGGLE_DATASET_TITLE,
    "id": KAGGLE_DATASET_ID,
    "licenses": [{"name": "unknown"}],
    "description": (
        "AIC keyframes L21-L30 transferred from Google Cloud Storage. "
        "Preserves original map-keyframes CSV files and includes reproducible "
        "file-level and frame-level metadata for multimodal vector embedding."
    ),
    "resources": [
        {"path": "dataset_summary.csv", "description": "Per-batch file counts and byte sizes."},
        {"path": "validation_report.csv", "description": "Per-video frame/map validation results."},
        {"path": "dataset_manifest.parquet", "description": "File-level GCS-to-Kaggle provenance manifest."},
        {"path": "frames_metadata.parquet", "description": "Frame-level temporal metadata from original map-keyframes."},
    ],
}

metadata_path = DATASET_DIR / "dataset-metadata.json"
metadata_path.write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Created:", DATASET_DIR / "README.md")
print("Created:", metadata_path)


## 10. Final local payload audit


In [ ]:
def directory_size(path: Path) -> int:
    return sum(p.stat().st_size for p in path.rglob("*") if p.is_file())

payload_bytes = directory_size(DATASET_DIR)
payload_gb = payload_bytes / 1024**3

top_level = sorted(DATASET_DIR.iterdir(), key=lambda p: p.name)
print("Top-level entries:", len(top_level))
for p in top_level:
    print(" -", p.name)

print(f"\nPayload size: {payload_gb:.2f} GB")
print(f"Free disk after download: {shutil.disk_usage('/kaggle/working').free/1024**3:.2f} GB")

for batch_id in BATCHES:
    root = DATASET_DIR / canonical_batch_dir(batch_id)
    key_root = root / "keyframes"
    map_root = root / "map-keyframes"

    videos = [p for p in key_root.iterdir() if p.is_dir()] if key_root.exists() else []
    frames = sum(
        1
        for v in videos
        for p in v.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )
    maps = list(map_root.glob("*.csv")) if map_root.exists() else []

    expected_frames = int(
        ((manifest_df["batch_id"] == batch_id) & (manifest_df["kind"] == "frame")).sum()
    )
    expected_maps = int(
        ((manifest_df["batch_id"] == batch_id) & (manifest_df["kind"] == "map")).sum()
    )

    assert frames == expected_frames, (batch_id, frames, expected_frames)
    assert len(maps) == expected_maps, (batch_id, len(maps), expected_maps)

    print(f"{batch_id}: videos={len(videos):,} frames={frames:,} maps={len(maps):,}")

if len(top_level) > 50:
    raise RuntimeError("More than 50 top-level entries. Restructure before Kaggle upload.")


## 11. Create / version Kaggle Dataset

- `--keep-tabular`: giữ Parquet.
- `--dir-mode tar`: Kaggle CLI đóng gói directory theo tar không nén để giảm overhead upload file nhỏ.
- Dataset mới private nếu `KAGGLE_PUBLIC=False`.


In [ ]:
import subprocess
import time

def kaggle_dataset_exists(dataset_id: str) -> bool:
    result = subprocess.run(
        ["kaggle", "datasets", "status", dataset_id],
        capture_output=True,
        text=True,
    )
    return result.returncode == 0

def upload_to_kaggle():
    if "/" not in KAGGLE_DATASET_ID:
        raise ValueError("Invalid KAGGLE_DATASET_ID.")

    if not VALIDATION_PATH.exists():
        raise RuntimeError("validation_report.csv missing; run validation first.")

    vdf = pd.read_csv(VALIDATION_PATH)
    if (vdf["status"] != "PASS").any():
        raise RuntimeError("Refusing Kaggle upload because validation has failures.")

    exists = kaggle_dataset_exists(KAGGLE_DATASET_ID)

    if exists:
        cmd = [
            "kaggle", "datasets", "version",
            "-p", str(DATASET_DIR),
            "-m", "Sync complete AIC keyframes and original map-keyframes from GCS",
            "-t",
            "-r", "tar",
        ]
    else:
        cmd = [
            "kaggle", "datasets", "create",
            "-p", str(DATASET_DIR),
            "-t",
            "-r", "tar",
        ]
        if KAGGLE_PUBLIC:
            cmd.append("--public")

    print("Running:", " ".join(cmd))
    t0 = time.time()
    subprocess.run(cmd, check=True)
    elapsed = time.time() - t0
    print(f"Kaggle CLI upload command finished in {elapsed/60:.2f} min")

    status = subprocess.run(
        ["kaggle", "datasets", "status", KAGGLE_DATASET_ID],
        capture_output=True,
        text=True,
    )
    print("Dataset status return code:", status.returncode)
    if status.stdout:
        print(status.stdout.strip())
    if status.stderr:
        print(status.stderr.strip())

if RUN_KAGGLE_UPLOAD:
    upload_to_kaggle()
else:
    print("Upload disabled: RUN_KAGGLE_UPLOAD=False")


## 12. SigLIP2 compatibility smoke test

Notebook embedding hiện tại tìm `*Keyframes_Lxx/keyframes/Lxx_Vyyy/*.jpg`, nên layout này tương thích trực tiếp.

Companion notebook v3 còn đọc CSV gốc trong sibling `map-keyframes/`.


In [ ]:
for batch_id in BATCHES:
    batch_root = DATASET_DIR / canonical_batch_dir(batch_id)
    keyframes_dir = batch_root / "keyframes"
    maps_dir = batch_root / "map-keyframes"

    assert keyframes_dir.is_dir(), keyframes_dir
    assert maps_dir.is_dir(), maps_dir

print("SigLIP2 input layout check: PASS")
print("Example:")
print(
    f"/kaggle/input/{KAGGLE_DATASET_SLUG}/"
    f"001_Keyframes_L21/keyframes/L21_V001/001.jpg"
)
